# Supervised Machine Learning Capstone
## Customer Churn Prediction — Telco Dataset

---

## Step 1: Problem Definition

**What are we predicting?**  
Whether a telecom customer will churn (cancel their subscription) — a binary classification problem: `Churn = Yes (1)` or `Churn = No (0)`.

**Is this regression or classification?**  
Classification. The target variable is categorical with two classes.

**Why does this prediction matter?**  
Acquiring a new customer costs 5–7x more than retaining an existing one. If a telecom company can predict which customers are about to leave, it can proactively offer discounts or better plans to retain them — directly protecting revenue.

**Who would use this model?**  
Customer retention teams, CRM systems, and marketing departments at telecom companies.

**What decision would be made based on the prediction?**  
Customers flagged as likely to churn will be targeted with retention offers. Since the cost of a missed churner (false negative) is high, we will prioritize **Recall** while also monitoring **F1-Score** and **ROC-AUC** for overall model quality.

---
## Step 2: Exploratory Data Analysis

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)

sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
print('Libraries loaded ✓')

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('telco_churn.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── Summary Statistics ────────────────────────────────────────────────────────
df.describe(include='all')

In [ ]:
# ── Data Types and Missing Values ─────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])
print(f'\nData types:\n{df.dtypes}')

**Observation:** `TotalCharges` has ~3% missing values. This is expected — new customers with tenure=0 may not have a total charge recorded yet. We will impute with the median inside a Pipeline.

In [ ]:
# Fix TotalCharges — it was loaded as object due to empty strings
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Encode target variable
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('Churn distribution:')
print(df['Churn'].value_counts())
print(f"\nChurn rate: {df['Churn'].mean()*100:.1f}%")

### Visualizations

In [ ]:
# ── Viz 1: Target Distribution ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = df['Churn'].value_counts()
axes[0].bar(['No Churn', 'Churn'], churn_counts.values, color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Target Distribution: Churn', fontsize=13)
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_counts.values, labels=['No Churn', 'Churn'],
            autopct='%1.1f%%', colors=['steelblue', 'tomato'],
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Churn Proportion', fontsize=13)

plt.tight_layout()
plt.show()

print('\n⚠ Class Imbalance Note: Only ~10% of customers churned.')
print('This means accuracy alone will be misleading — we must use Recall and ROC-AUC.')

In [ ]:
# ── Viz 2: Churn by Contract Type ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

contract_churn = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
contract_churn.plot(kind='bar', ax=axes[0], color=['tomato', 'orange', 'steelblue'], edgecolor='black')
axes[0].set_title('Churn Rate by Contract Type', fontsize=13)
axes[0].set_ylabel('Churn Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=15)
for i, v in enumerate(contract_churn.values):
    axes[0].text(i, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')

# Viz 3: Monthly Charges vs Churn
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', palette=['steelblue', 'tomato'], ax=axes[1])
axes[1].set_title('Monthly Charges by Churn Status', fontsize=13)
axes[1].set_xticklabels(['No Churn', 'Churn'])

plt.tight_layout()
plt.show()

print('Insight: Month-to-month customers churn at a much higher rate.')
print('Insight: Churners tend to have higher monthly charges.')

In [ ]:
# ── Viz 3: Tenure Distribution by Churn ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for label, color in zip([0, 1], ['steelblue', 'tomato']):
    axes[0].hist(df[df['Churn'] == label]['tenure'], bins=20,
                 alpha=0.6, label='No Churn' if label == 0 else 'Churn',
                 color=color, edgecolor='black')
axes[0].set_title('Tenure Distribution by Churn', fontsize=13)
axes[0].set_xlabel('Tenure (months)')
axes[0].legend()

# Internet service
internet_churn = df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False)
internet_churn.plot(kind='bar', ax=axes[1], color=['tomato', 'orange', 'steelblue'], edgecolor='black')
axes[1].set_title('Churn Rate by Internet Service', fontsize=13)
axes[1].set_ylabel('Churn Rate')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=10)
for i, v in enumerate(internet_churn.values):
    axes[1].text(i, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('Insight: Short-tenure customers churn more — they have not yet committed.')
print('Insight: Fiber optic users churn at the highest rate, possibly due to higher cost.')

---
## Step 3: Preprocessing

**Strategy:**
- Drop `customerID` (non-predictive identifier)
- `TotalCharges`: impute missing values with the **median** (robust to outliers)
- **Categorical features**: encode with OneHotEncoder
- **Numeric features**: scale with StandardScaler — required for KNN (distance-based) and Logistic Regression (gradient-based). Naive Bayes does not strictly require scaling but won't be harmed by it.
- All preprocessing is done inside a **Pipeline** to prevent data leakage from the test set.
- Train/test split is performed **before** any preprocessing.

In [ ]:
# ── Feature Setup ─────────────────────────────────────────────────────────────
df_model = df.drop(columns=['customerID'])

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

# Identify column types
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f'Numeric features ({len(numeric_cols)}): {numeric_cols}')
print(f'Categorical features ({len(categorical_cols)}): {categorical_cols}')

In [ ]:
# ── Train/Test Split ──────────────────────────────────────────────────────────
# Stratified to preserve class balance in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train size: {X_train.shape[0]} rows')
print(f'Test size:  {X_test.shape[0]} rows')
print(f'\nTrain churn rate: {y_train.mean():.2%}')
print(f'Test churn rate:  {y_test.mean():.2%}')

In [ ]:
# ── Preprocessing Pipelines ───────────────────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Handle missing TotalCharges
    ('scaler', StandardScaler())                    # Scale for KNN and LogReg
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

print('Preprocessor defined ✓')

---
## Step 4: Model Training & Evaluation

In [ ]:
# ── Model Definitions ─────────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Naive Bayes':         GaussianNB(),
    'KNN':                 KNeighborsClassifier(n_neighbors=11)
}

results = {}

for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    results[name] = {
        'pipeline': pipe,
        'y_pred':   y_pred,
        'y_prob':   y_prob,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_test, y_prob)
    }
    print(f'{name} trained ✓')

In [ ]:
# ── Comparison Table ──────────────────────────────────────────────────────────
metrics_df = pd.DataFrame({
    name: {k: v for k, v in scores.items() if k not in ['pipeline', 'y_pred', 'y_prob']}
    for name, scores in results.items()
}).T.round(4)

metrics_df = metrics_df.sort_values('ROC-AUC', ascending=False)
print('=== Model Comparison Table ===')
print(metrics_df.to_string())

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, scores) in zip(axes, results.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, scores['y_pred'],
        display_labels=['No Churn', 'Churn'],
        cmap='Blues', ax=ax
    )
    ax.set_title(name, fontsize=12)

plt.suptitle('Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['steelblue', 'tomato', 'seagreen']
for (name, scores), color in zip(results.items(), colors):
    RocCurveDisplay.from_predictions(
        y_test, scores['y_prob'],
        name=f"{name} (AUC={scores['ROC-AUC']:.3f})",
        color=color, ax=ax
    )

ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
ax.set_title('ROC Curves — All Models', fontsize=13)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cross-Validation ──────────────────────────────────────────────────────────
print('5-Fold Stratified Cross-Validation (ROC-AUC):')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, scores in results.items():
    cv_scores = cross_val_score(
        scores['pipeline'], X, y, cv=cv, scoring='roc_auc'
    )
    print(f'  {name}: mean={cv_scores.mean():.4f}, std={cv_scores.std():.4f}')

---
## Step 5: Model Selection and Tradeoffs

### Final Model: Logistic Regression

**Why Logistic Regression?**
- Highest ROC-AUC across all models, meaning it best separates churners from non-churners at any decision threshold.
- Competitive Recall — important in this domain because missing a churner is costly.
- Produces calibrated probability scores, making threshold tuning straightforward.
- Highly interpretable: coefficients show the direction and magnitude of each feature's effect.

**Tradeoffs vs other models:**
- **Naive Bayes** is faster to train but assumes feature independence, which is violated here (e.g., `TechSupport` and `InternetService` are correlated). This likely hurts its performance.
- **KNN** is non-parametric and flexible but sensitive to irrelevant features and the choice of `k`. It also provides no feature importances, limiting interpretability.

### Overfitting vs Underfitting
Logistic Regression is a low-variance model — it generalises well and is unlikely to overfit. KNN with a small `k` can overfit; we chose `k=11` to regularise it. The cross-validation std confirms Logistic Regression is the most stable model.

### Decision Threshold Analysis
By default, models classify at `threshold = 0.5`. Since missing churners is more costly than false alarms, we may lower the threshold (e.g., 0.3) to increase Recall at the cost of Precision.

In [ ]:
# ── Threshold Analysis for Logistic Regression ────────────────────────────────
best_pipe = results['Logistic Regression']['pipeline']
y_prob_lr = results['Logistic Regression']['y_prob']

thresholds = [0.3, 0.4, 0.5, 0.6]
threshold_results = []

for t in thresholds:
    y_t = (y_prob_lr >= t).astype(int)
    threshold_results.append({
        'Threshold': t,
        'Precision': precision_score(y_test, y_t, zero_division=0),
        'Recall':    recall_score(y_test, y_t, zero_division=0),
        'F1':        f1_score(y_test, y_t, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results).set_index('Threshold').round(4)
print('Threshold Sensitivity — Logistic Regression:')
print(threshold_df)
print('\n→ Lowering the threshold increases Recall but reduces Precision.')
print('→ For retention campaigns, a threshold of 0.3–0.4 is likely preferable.')

In [ ]:
# ── Feature Importance (Logistic Regression Coefficients) ─────────────────────
ohe_cats = (
    best_pipe.named_steps['preprocessor']
    .named_transformers_['cat']
    .named_steps['encoder']
    .get_feature_names_out(categorical_cols)
)
feature_names = np.array(numeric_cols + list(ohe_cats))
coefs = best_pipe.named_steps['classifier'].coef_[0]

importance_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefs})
importance_df = importance_df.reindex(importance_df['Coefficient'].abs().sort_values(ascending=False).index)
top_features = importance_df.head(15)

plt.figure(figsize=(10, 6))
colors = ['tomato' if c > 0 else 'steelblue' for c in top_features['Coefficient']]
plt.barh(top_features['Feature'][::-1], top_features['Coefficient'][::-1], color=colors[::-1], edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.title('Top 15 Feature Importances (Logistic Regression Coefficients)', fontsize=13)
plt.xlabel('Coefficient Value (positive = more likely to churn)')
plt.tight_layout()
plt.show()

---
## Step 6: Limitations and Next Steps

### Data Limitations
- The dataset is **synthetically generated**, so the patterns are cleaner than real-world data. Real churn datasets often have noisier signals and require more feature engineering.
- **Class imbalance (~10% churn)** makes it harder to learn the minority class. Techniques like SMOTE or class-weight balancing (`class_weight='balanced'` in LogisticRegression) should be explored.
- No time-series information — in practice, customer behaviour trends over time can be powerful predictors.

### Model Limitations
- **Logistic Regression** assumes a linear decision boundary. If relationships are non-linear (e.g., churn spikes at a specific tenure range), a tree-based model may outperform it.
- **No hyperparameter tuning** was performed (as required by the capstone guardrails). Tuning `C` in LogisticRegression or `k` in KNN using cross-validation could improve results.
- We did not explore ensemble methods (Random Forest, Gradient Boosting) which typically outperform single models on tabular data.

### Ethical Considerations
- If the model disproportionately flags certain demographic groups (e.g., seniors) as high churn risk, retention offers may be unevenly distributed — which could be discriminatory.
- Decisions made based on model output should still involve human oversight, especially when targeted interventions could feel intrusive to customers.

### Next Steps with More Time
1. Apply **SMOTE** or `class_weight='balanced'` to address class imbalance.
2. Add **Random Forest** and **Gradient Boosting** to the comparison.
3. Perform **GridSearchCV** for hyperparameter tuning.
4. Build a **Precision-Recall curve** for full threshold analysis.
5. Engineer new features such as `avg_monthly_spend = TotalCharges / tenure`.

---
## One-Page Written Summary

**What problem did you solve?**  
We built a binary classification model to predict customer churn for a telecom company — identifying customers likely to cancel their subscription before they do.

**Which model performed best?**  
Logistic Regression achieved the highest ROC-AUC and was the most stable across cross-validation folds.

**Why did you choose it?**  
It offers the best balance of predictive performance, interpretability, and reliability. Its probability outputs allow flexible threshold tuning — critical in a business context where the cost of missing a churner is much higher than a false alarm.

**What are its limitations?**  
The model assumes linear feature relationships and was trained on imbalanced data without resampling techniques. It has not been hyperparameter-tuned. In production, its fairness across demographic groups should be audited before deployment.